# Telco customer churn: final evaluation of the frozen plan

## Goal
From the frozen plan to the final metrics: load the recipe, verify its configuration, reproduce the split, fit, predict, and compare with the saved metrics.

The original test set was looked at once during the first baseline, so it is not a fully untouched holdout. The results here are not used to choose a model or a threshold.

## Setup
The frozen plan is a JSON record of the chosen threshold, features and model settings. It is not a model.

In [1]:
from pathlib import Path
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.base import clone
from sklearn.exceptions import NotFittedError
from sklearn.model_selection import train_test_split
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)

# Project folder = the nearest parent that contains reports/final-plan.json
PROJECT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "reports" / "final-plan.json").exists())

plan = json.loads((PROJECT / "reports/final-plan.json").read_text())
print("Plan frozen:", plan["frozen"])
print("Fixed threshold:", plan["threshold"])

Plan frozen: True
Fixed threshold: 0.409


## 1. Pre-checks
Nothing is trained or predicted in this step. `plan` is the written record; `recipe` is the unfitted Pipeline saved with joblib; `final_pipeline` is a copy of the recipe that will be fitted later.

In [2]:
# The plan must be frozen and the development checks passed
assert plan["frozen"] is True
assert plan["development_checks_passed"] is True
assert plan["frozen_at"] and plan["selection_reason"]
assert 0 < plan["threshold"] < 1

# The SHA-256 of the recipe file must match the frozen record
recipe_path = PROJECT / plan["recipe_artifact"]
if not recipe_path.is_file():
    raise FileNotFoundError(f"Frozen pipeline recipe not found: {recipe_path}")
recipe_hash = hashlib.sha256(recipe_path.read_bytes()).hexdigest()
assert recipe_hash == plan["recipe_sha256"], "recipe file does not match the frozen record"

# joblib.load restores the saved recipe; no fit happens here
recipe = joblib.load(recipe_path)
preprocessor = recipe.named_steps["preprocess"]
model = recipe.named_steps["model"]

# Column routing, model type and parameters must equal the plan
feature_routes = {name: list(columns) for name, _, columns in preprocessor.transformers}
assert feature_routes == plan["features"]
assert model.__class__.__name__ == plan["model_parameters"]["estimator"]
for parameter in ("max_iter", "solver", "C", "class_weight"):
    assert model.get_params()[parameter] == plan["model_parameters"][parameter]

# Preprocessing steps: median imputation, StandardScaler, encoder ignoring unknown categories
numeric_pipeline = preprocessor.transformers[0][1]
categorical_pipeline = preprocessor.transformers[1][1]
assert numeric_pipeline.named_steps["imputer"].strategy == "median"
assert numeric_pipeline.named_steps["scaler"].__class__.__name__ == "StandardScaler"
assert categorical_pipeline.named_steps["encoder"].handle_unknown == "ignore"

# The saved recipe must be unfitted
try:
    check_is_fitted(recipe)
except NotFittedError:
    pass
else:
    raise AssertionError("the recipe is already fitted; an unfitted recipe is expected")

# clone gives a new unfitted Pipeline with the same settings; only this copy is fitted
final_pipeline = clone(recipe)
expected_features = plan["features"]["numeric"] + plan["features"]["categorical"]
assert len(expected_features) == len(set(expected_features)) == 19
print("Pre-checks passed: unfitted recipe, 19 features, model parameters and SHA-256 all match the frozen record.")

Pre-checks passed: unfitted recipe, 19 features, model parameters and SHA-256 all match the frozen record.


## 2. Reproducing the original 80/20 split
Same CSV; `Churn` and `customerID` removed; `TotalCharges` converted to numeric; `random_state=2002`, `stratify=y`. The split decides who is used for fitting and who is held out; it is not part of the Pipeline. Only `x_train / y_train` are used to fit, and `x_test / y_test` are kept for evaluation.

In [3]:
data_path = PROJECT / "data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"
if not data_path.is_file():
    raise FileNotFoundError(f"Raw data not found: {data_path}")

df = pd.read_csv(data_path)
assert df["customerID"].is_unique
y = df["Churn"].copy()
x = df.drop(columns=["Churn", "customerID"]).copy()
# Blanks become NaN and are imputed by the recipe
x["TotalCharges"] = pd.to_numeric(x["TotalCharges"], errors="coerce")
assert len(x.columns) == len(expected_features)
assert set(x.columns) == set(expected_features)
assert x.index.equals(y.index)

x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.2, random_state=2002, stratify=y
)
assert len(x_train) == len(y_train) == 5634
assert len(x_test) == len(y_test) == 1409
# Features and target aligned; no customer on both sides
assert x_train.index.equals(y_train.index)
assert x_test.index.equals(y_test.index)
assert x_train.index.intersection(x_test.index).empty
print("Split reproduced: train", len(x_train), "| test", len(x_test))

Split reproduced: train 5634 | test 1409


## 3. Final evaluation
`fit` learns from the training set; `predict_proba` returns the `No` and `Yes` probabilities for each test customer; the frozen threshold 0.409 turns the `Yes` probability into a label. The threshold is not revised in the light of these results.

In [4]:
# fit learns the imputation values, the encoded categories and the model coefficients from the training set
final_pipeline.fit(x_train, y_train)
class_order = list(final_pipeline.named_steps["model"].classes_)
assert set(class_order) == {"No", "Yes"}
yes_col = class_order.index("Yes")
test_yes_prob = final_pipeline.predict_proba(x_test)[:, yes_col]
assert len(test_yes_prob) == len(x_test) == 1409
assert np.isfinite(test_yes_prob).all()
assert ((0 <= test_yes_prob) & (test_yes_prob <= 1)).all()

# Only the threshold frozen in the plan is used
test_pred = np.where(test_yes_prob >= plan["threshold"], "Yes", "No")
# Rows are actual No/Yes, columns predicted No/Yes; ravel gives TN, FP, FN, TP
test_cm = confusion_matrix(y_test, test_pred, labels=["No", "Yes"])
tn, fp, fn, tp = (int(value) for value in test_cm.ravel())
assert tn + fp + fn + tp == len(x_test)

test_metrics = {
    "threshold": plan["threshold"],
    "frozen_at": plan["frozen_at"],
    "recipe_sha256": recipe_hash,
    "original_test_previously_viewed": plan["original_test_previously_viewed"],
    "train_count": len(x_train),
    "test_count": len(x_test),
    "class_order": class_order,
    "actual_yes": int(y_test.eq("Yes").sum()),
    "predicted_yes": int((test_pred == "Yes").sum()),
    "TN": tn, "FP": fp, "FN": fn, "TP": tp,
    "yes_precision": float(precision_score(y_test, test_pred, pos_label="Yes", zero_division=0)),
    "yes_recall": float(recall_score(y_test, test_pred, pos_label="Yes", zero_division=0)),
    "yes_f1": float(f1_score(y_test, test_pred, pos_label="Yes", zero_division=0)),
    "accuracy": float(accuracy_score(y_test, test_pred)),
    # AUC measures ranking, so it takes the Yes probabilities, not the thresholded labels
    "roc_auc": float(roc_auc_score(y_test.eq("Yes"), test_yes_prob)),
}

display(pd.DataFrame(test_cm, index=["Actual No", "Actual Yes"], columns=["Predicted No", "Predicted Yes"]))
print(classification_report(y_test, test_pred, labels=["No", "Yes"], zero_division=0))
print("Test ROC-AUC:", test_metrics["roc_auc"])
print("Fixed threshold:", plan["threshold"], "| predicted Yes:", test_metrics["predicted_yes"])

,Predicted No,Predicted Yes
Actual No,882,153
Actual Yes,127,247


              precision    recall  f1-score   support

          No       0.87      0.85      0.86      1035
         Yes       0.62      0.66      0.64       374

    accuracy                           0.80      1409
   macro avg       0.75      0.76      0.75      1409
weighted avg       0.81      0.80      0.80      1409

Test ROC-AUC: 0.8385982588028623
Fixed threshold: 0.409 | predicted Yes: 400


## Checks
Internal consistency of the numbers, then a comparison with the metrics saved in `reports/final-metrics.json` when the evaluation was first run. This notebook does not write that file: a rerun reproduces the saved record or fails.

In [5]:
assert x_test.index.equals(y_test.index)
assert test_metrics["test_count"] == 1409
assert test_metrics["TN"] + test_metrics["FP"] + test_metrics["FN"] + test_metrics["TP"] == 1409
# Actual Yes = TP + FN; predicted Yes = TP + FP
assert test_metrics["actual_yes"] == test_metrics["TP"] + test_metrics["FN"]
assert test_metrics["predicted_yes"] == test_metrics["TP"] + test_metrics["FP"]
assert test_metrics["threshold"] == plan["threshold"]
assert test_metrics["recipe_sha256"] == plan["recipe_sha256"]
assert np.isclose(test_metrics["yes_recall"], test_metrics["TP"] / test_metrics["actual_yes"])

saved_metrics = json.loads((PROJECT / "reports/final-metrics.json").read_text())
for key, value in test_metrics.items():
    saved = saved_metrics[key]
    same = np.isclose(value, saved) if isinstance(value, float) else value == saved
    assert same, f"{key}: {value} differs from the saved {saved}"
print("All", len(test_metrics), "values match reports/final-metrics.json.")

All 18 values match reports/final-metrics.json.


## Result
With the frozen threshold of 0.409, the model flags 400 of the 1,409 test customers. 247 of them churned and 153 did not; 127 of the 374 actual churners are missed.

| Test metric | Result |
|---|---:|
| TN / FP / FN / TP | 882 / 153 / 127 / 247 |
| Churn-class precision / recall / F1 | 0.6175 / 0.6604 / 0.6382 |
| Accuracy / ROC-AUC | 0.8013 / 0.8386 |

The validation F1 was 0.6745 and the test F1 is 0.6382. The original test set was looked at once during the first baseline, so this is not performance on fully unseen data, and it is not used to revise the model or the threshold.